# HuBERT Hooked Audio Sanity Test

This notebook runs basic checks on `HookedAudioEncoder`: forward pass, determinism, gradient flow, and an optional comparison to Hugging Face HuBERT.

In [1]:
# Install dependencies and the forked repo
!pip -q install -U pip setuptools wheel
!git clone https://github.com/david-wei-01001/TransformerLens.git
%cd TransformerLens
!pip -q install -e .
print("\n⚠️ IMPORTANT: Restart runtime now, then run the next cell.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
Cloning into 'TransformerLens'...
remote: Enumerating objects: 5328, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 5328 (delta 126), reused 78 (delta 67), pack-reused 5138 (from 2)
Receiving objects: 100% (5328/5328), 25.07 MiB | 6.86 MiB/s, done.
Resolving deltas: 100% (3580/3580), done.
/content/TransformerLens
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing bu

In [1]:
import math
from typing import Any

import numpy as np
import torch

from transformer_lens import HookedAudioEncoder

SAMPLE_RATE = 16000
DURATION_S = 1.0
BATCH_SIZE = 1
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
HF_CHECKPOINT = 'facebook/hubert-base-ls960'

torch.set_grad_enabled(True)
print('Device:', DEVICE)

Device: cuda


In [2]:
def make_sine(
    frequency: float = 440.0,
    sr: int = SAMPLE_RATE,
    duration: float = DURATION_S,
    amplitude: float = 0.1,
) -> np.ndarray:
    t = np.linspace(0, duration, int(sr * duration), endpoint=False, dtype=np.float32)
    wav = amplitude * np.sin(2 * math.pi * frequency * t)
    return wav


def extract_tensor(output: Any) -> torch.Tensor:
    if isinstance(output, torch.Tensor):
        return output
    if isinstance(output, dict):
        for key in ('predictions', 'logits', 'ctc_logits'):
            if key in output and isinstance(output[key], torch.Tensor):
                return output[key]
    raise TypeError(f'Could not extract tensor from output type {type(output)}')

In [3]:
def run_basic_sanity_tests(model, waveform_np):
    """Run quick checks: forward pass, shape, finite, deterministic, grad flow."""
    model.to(DEVICE)

    x = torch.from_numpy(waveform_np).unsqueeze(0).to(DEVICE)

    # 1) Eval forward: no grad
    model.eval()
    with torch.no_grad():
        out1 = model(x)
    print('Forward (eval) output type:', type(out1))
    out_tensor = extract_tensor(out1)

    print('Output shape:', tuple(out_tensor.shape))
    print(
        'Output stats: min=%.6g max=%.6g mean=%.6g'
        % (out_tensor.min().item(), out_tensor.max().item(), out_tensor.mean().item())
    )
    assert torch.isfinite(out_tensor).all(), 'Found NaNs or Infs in forward output!'

    # 2) Determinism in eval
    with torch.no_grad():
        out2 = model(x)
    out2_tensor = extract_tensor(out2)
    if not torch.allclose(out_tensor, out2_tensor, atol=1e-6):
        print(
            'Warning: outputs differ between two eval runs (non-deterministic?), max diff:',
            (out_tensor - out2_tensor).abs().max().item(),
        )
    else:
        print('Determinism test passed (eval mode).')

    # 3) Gradient flow test in train mode
    model.train()
    for p in model.parameters():
        if p.grad is not None:
            p.grad.detach_()
            p.grad.zero_()
    out_train = model(x)
    out_train_tensor = extract_tensor(out_train)
    loss = out_train_tensor.mean()
    loss.backward()

    grads_found = any(
        (p.grad is not None and torch.isfinite(p.grad).all())
        for p in model.parameters()
        if p.requires_grad
    )
    assert grads_found, 'No finite gradients found on any parameter after backward()'
    print('Gradient check passed: some parameters have finite gradients.')

In [4]:
def optional_compare_to_hf(your_model, waveform_np, sr: int = SAMPLE_RATE):
    """
    OPTIONAL: compare your_model outputs to Hugging Face's HubertModel outputs.
    Requires transformers and internet access.
    """
    try:
        from transformers import HubertModel, Wav2Vec2FeatureExtractor
    except Exception as e:
        print('Transformers or feature extractor not available:', e)
        return

    print('Loading Hugging Face HubertModel for optional comparison...')
    hf_feat = Wav2Vec2FeatureExtractor(sampling_rate=sr, do_normalize=True)
    hf_model = HubertModel.from_pretrained(HF_CHECKPOINT).to(DEVICE).eval()

    input_values = hf_feat(waveform_np, sampling_rate=sr, return_tensors='pt').get('input_values')
    input_values = input_values.to(DEVICE)

    with torch.no_grad():
        hf_outputs = hf_model(input_values).last_hidden_state
    hf_embedding = hf_outputs.mean(dim=1)

    your_model.eval()
    with torch.no_grad():
        your_out = your_model(torch.from_numpy(waveform_np).unsqueeze(0).to(DEVICE))
    your_tensor = extract_tensor(your_out)

    if your_tensor.ndim == 3:
        your_emb = your_tensor.mean(dim=1)
    else:
        your_emb = your_tensor

    if hf_embedding.shape[1] != your_emb.shape[1]:
        print(
            f'Dimension mismatch (HF {hf_embedding.shape[1]} vs your {your_emb.shape[1]}). '
            'Compare after projecting to a common dimension if needed.'
        )
        return

    cos = torch.nn.functional.cosine_similarity(hf_embedding, your_emb, dim=1)
    print('Cosine similarity between HF pooled embedding and your model:', cos.cpu().numpy())

In [5]:
# Create sample waveform
wav = make_sine(frequency=440.0, sr=SAMPLE_RATE, duration=DURATION_S)

# Instantiate your model
model = HookedAudioEncoder.from_pretrained('facebook/hubert-base-ls960').to(DEVICE)

# Run tests
run_basic_sanity_tests(model, wav)

If using HuBERT for interpretability research, keep in mind that HuBERT has some significant architectural differences to GPT. For example, LayerNorms are applied *after* the attention and MLP components, meaning that the last LayerNorm in a block cannot be folded.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

Moving model to device:  cuda
Loaded pretrained model facebook/hubert-base-ls960 into HookedEncoder
Moving model to device:  cuda
Moving model to device:  cuda
Forward (eval) output type: <class 'torch.Tensor'>
Output shape: (1, 49, 768)
Output stats: min=-2.88404 max=2.90499 mean=-0.00641759
Determinism test passed (eval mode).
Gradient check passed: some parameters have finite gradients.


In [6]:
# Optional comparison to Hugging Face (requires transformers + internet)
optional_compare_to_hf(model, wav, sr=SAMPLE_RATE)

Loading Hugging Face HubertModel for optional comparison...
Cosine similarity between HF pooled embedding and your model: [0.99999994]
